# 例 6.3 · 用 Kelly 准则计算 OIH、RKH 和 RTH 的仓位

本例复现 Ernest Chan 的 [MATLAB 示例](../matlab/example6_3.m) 和对应的 [Python 实现](../python/examples/example6_3.py)。用三只资产的历史复权收盘价估计超额收益与协方差，再求出使模型中的组合增长率最大的仓位。

全篇按 OIH、RKH、RTH 排列资产。每年按 252 个交易日计算，年化无风险利率取 4%。

In [ ]:
import sys
from pathlib import Path

import numpy as np

repo_root = next(
    base for base in (Path.cwd(), *Path.cwd().parents)
    if (base / 'python' / 'epchan').is_dir()
)
sys.path.insert(0, str(repo_root / 'python'))

from epchan.get_tday import get_tday
from epchan.lag1 import lag1

SYMBOLS = ('OIH', 'RKH', 'RTH')
TRADING_DAYS = 252
RISK_FREE_RATE = 0.04

## 1. 按交易日对齐复权收盘价

取三个文件中交易日的并集，按日期升序排列。每行对应一个交易日，每列对应一只资产，缺失价格保留为 `NaN`。

In [ ]:
series = [get_tday(repo_root / 'data' / f'{symbol}.xls') for symbol in SYMBOLS]
trading_dates = np.unique(np.concatenate([dates for dates, _ in series]))
prices = np.full((len(trading_dates), len(SYMBOLS)), np.nan)
for column, (dates, closes) in enumerate(series):
    prices[np.searchsorted(trading_dates, dates), column] = closes

print(f'价格矩阵覆盖 {len(trading_dates)} 个交易日')

## 2. 估计年化超额收益 M 和协方差矩阵 C

先用相邻两行的价格计算日收益，再剔除任一资产收益无效的行。这个顺序可以避免把跨越缺失日期的涨跌当作单日收益。每天扣除 `0.04 / 252`，再将日均超额收益和日样本协方差分别乘以 252。

In [ ]:
previous = lag1(prices)
returns = (prices - previous) / previous
valid = np.isfinite(returns).all(axis=1)
excess = returns[valid] - RISK_FREE_RATE / TRADING_DAYS
m = TRADING_DAYS * excess.mean(axis=0)
c = TRADING_DAYS * np.cov(excess, rowvar=False, ddof=1)

print(f'共同有效收益天数 = {len(excess)}')
print(f'M = {m.round(4)}')
print(f'C =\n{c.round(4)}')

## 3. 求 Kelly 仓位和组合指标

求解 $CF=M$ 即可得到 Kelly 仓位。$F_i$ 表示每单位组合净资产在该标的上的头寸，正值为做多，负值为做空。

组合的年化方差为 $v=F^T C F$。在 Kelly 解处，组合的年化超额收益 $F^T M=v$，因此书中模型给出的最大年化复合增长率为 $g=r_f+v/2$，夏普比率为 $S=F^T M/\sqrt{v}=\sqrt{v}$。

In [ ]:
f = np.linalg.solve(c, m)
variance = f @ c @ f
g = RISK_FREE_RATE + variance / 2
s = np.sqrt(variance)

for symbol, position in zip(SYMBOLS, f):
    print(f'{symbol} 仓位 = {position:+.4f}')
print(f'模型估计的最大年化复合增长率 = {g:.2%}')
print(f'组合夏普比率 = {s:.4f}')